# 1D Steady-State Heat Conduction: Finite Difference Method (FDM)

This notebook models the 1D steady-state heat conduction across a composite wall (such as a rocket engine wall) using the **Finite Difference Method (FDM)**. 

By discretizing continuous materials into distinct "nodes," we can transform differential calculus into a system of linear algebraic equations, which are then solved simultaneously using Python, `numPy`, and `scipy`.

## 1. The Physics: Fourier's Law
In a steady-state system with no internal heat generation, the 1D heat equation derived from Fourier's Law is:

$$ \frac{d}{dx} \left( k \frac{dT}{dx} \right) = 0 $$

For a uniform material where thermal conductivity ($k$) is constant, the temperature gradient is linear:

$$ \frac{d^2T}{dx^2} = 0 $$

## 2. Mathematical Discretization
Computers solve discrete points, not continuous curves. We divide the wall into nodes separated by a physical distance $\Delta x$. Using the **Central Difference Approximation** from Taylor series, the second derivative at any internal node $i$ becomes:

$$ \frac{T_{i-1} - 2T_i + T_{i+1}}{\Delta x^2} = 0 \implies T_{i-1} - 2T_i + T_{i+1} = 0 $$

## 3. Composite Walls & Interface Nodes
When two different materials meet, we merge their boundaries into a single **shared interface node**. At this precise coordinate, the thermal conductivity changes abruptly. 

To determine the temperature at this junction, we apply the principle of **Continuity** and **Conservation of Energy** (heat leaving Layer 1 exactly equals heat entering Layer 2):

$$ k_1 \frac{T_{i-1} - T_i}{\Delta x_1} = k_2 \frac{T_i - T_{i+1}}{\Delta x_2} $$

Letting thermal conductance $C = \frac{k}{\Delta x}$, the algebraic equation for an interface node becomes:
$$ C_1 T_{i-1} - (C_1 + C_2) T_i + C_2 T_{i+1} = 0 $$

## 4. The Global Matrix
We assemble these algebraic relationships into a global matrix equation of the form $A \cdot \mathbf{T} = \mathbf{b}$. 

*   **Matrix $A$**: A tridiagonal matrix storing the node coefficients ($1, -2, 1$ for internal nodes, and $C_1, -(C_1+C_2), C_2$ for interfaces).
*   **Vector $\mathbf{b}$**: Stores the known boundary temperatures (e.g., hot gas inside, ambient air outside).
*   **Vector $\mathbf{T}$**: The unknown temperatures at every node, which we solve for using `scipy.linalg.solve`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve


In [ ]:
no_mat=int(input("Enter the number of materials: "))
list_storage=[]
list_nodes=[]
for i in range(no_mat):
    print("Material ",i+1,":\n")
    mat_name=int(input("Enter the number of nodes: "))
    k=float(input("Enter the thermal conductivity of material (W/mK): "))
    l=float(input("Enter the length of material (m): "))
    list_storage.append([mat_name,k,l])
    list_nodes.append(mat_name)

total_nodes=sum(list_nodes)-(len(list_storage)-1)
print(f"Total number of nodes (N): {total_nodes}")

In [ ]:
int_temp=float(input("Enter the interior temperature (K): "))
ext_temp=float(input("Enter the exterior temperature (K): "))

temp_mat=np.zeros((total_nodes,1))
temp_mat[0,0]=int_temp
temp_mat[-1,0]=ext_temp

In [ ]:
interface_constants=[i[1]/(i[2]/(i[0]-1)) for i in list_storage]

In [ ]:
solver_mat=np.zeros((total_nodes, total_nodes))
solver_mat[0,0]=1
solver_mat[-1,-1]=1
count=1
for i in range(1,total_nodes-1):
    if(count==list_nodes[0]-1):
        solver_mat[i,i-1]=interface_constants[0]
        solver_mat[i,i]=-(interface_constants[0]+interface_constants[1])
        solver_mat[i,i+1]=interface_constants[1]
        count=1
        list_nodes.pop(0)
        interface_constants.pop(0)
    else:
        solver_mat[i,i-1]=1
        solver_mat[i,i]=-2
        solver_mat[i,i+1]=1
        count+=1


In [ ]:
T = solve(solver_mat, temp_mat)
x_global = []
current_distance = 0.0

for i, layer in enumerate(list_storage):
    n = layer[0]
    length = layer[2]
    x_layer = np.linspace(current_distance, current_distance + length, n)
    
    if i == 0:
        x_global.extend(x_layer)
    else:
        x_global.extend(x_layer[1:])
    
    current_distance += length
print("Temperature at different nodes (K):")
for i, temp in enumerate(T):
    print(f"Node {i}: \n---> Distance from middle: {x_global[i]:.2f} m\n---> Temperature {temp[0]} K\n")
x_global = np.array(x_global)


In [ ]:

plt.figure(figsize=(10, 6))
plt.plot(x_global, T, color='black', linestyle=':', label='Temperature Drop')

pivot_x = [x_global[0]]
pivot_T = [T[0]]

current_idx = 0
colors = ['red', 'blue', 'green', 'orange']

for idx, i in enumerate(list_storage):
    n = i[0]
    x_layer = x_global[current_idx : current_idx + n]
    T_layer = T[current_idx : current_idx + n]
    
    plt.scatter(x_layer[1:-1], T_layer[1:-1], color=colors[idx % len(colors)], label=f'Material {idx+1} (k)={i[1]} W/mK')
    
    pivot_x.append(x_layer[-1])
    pivot_T.append(T_layer[-1])
    
    current_idx += (n - 1)

plt.scatter(pivot_x, pivot_T, color='black', s=80, zorder=3, label='Pivots/Boundaries')

plt.title('Temperature Profile by Material')
plt.xlabel('Distance (m)')
plt.ylabel('Temperature (K)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
